In [3]:
from pathlib import Path
import shutil
import time
import json

import pandas as pd
from openpyxl import load_workbook

# --- Adjust this path if needed ---
from pathlib import Path

REPO_ROOT = Path.cwd().parent  # notebook is in tests/, repo root is one level up
TEMPLATE = REPO_ROOT / "resources" / "templates" / "Resources.xlsm"

assert TEMPLATE.exists(), f"Template not found at {TEMPLATE}"
print("Repo root:", REPO_ROOT)
print("Template:", TEMPLATE)

assert TEMPLATE.exists(), f"Template not found at {TEMPLATE}"

# Import your validator runner
# (adjust the import if your function name/module differs)
from excel2sbol.validator import run_sheet_validator


def fresh_copy(tag: str) -> Path:
    """Copy the template workbook to a unique temp file for a test."""
    out_dir = REPO_ROOT / "tests" / "_tmp_validator_runs"
    out_dir.mkdir(parents=True, exist_ok=True)
    ts = int(time.time() * 1000)
    out_path = out_dir / f"{tag}_{ts}.xlsm"
    shutil.copy(TEMPLATE, out_path)
    return out_path


def run_v(path: Path, echo: bool = False) -> dict:
    """Run validator and return result dict."""
    return run_sheet_validator(str(path), validate_only=True, echo=echo)


def show(result: dict, max_items: int = 10):
    print("ok:", result.get("ok"))
    print("errors:", len(result.get("errors", [])))
    print("warnings:", len(result.get("warnings", [])))

    for e in result.get("errors", [])[:max_items]:
        print("ERROR:", e["message"])
    for w in result.get("warnings", [])[:max_items]:
        print("WARN :", w["message"])

def assert_has_error(result: dict, code: str):
    codes = [e["code"] for e in result.get("errors", [])]
    assert code in codes, f"Expected ERROR code {code}, got {codes}"


def assert_has_warning(result: dict, code: str):
    codes = [w["code"] for w in result.get("warnings", [])]
    assert code in codes, f"Expected WARNING code {code}, got {codes}"


def assert_ok(result: dict):
    assert result.get("ok") is True, f"Expected ok=True, got {result.get('ok')}"


def column_definitions_df(path: Path) -> pd.DataFrame:
    df = pd.read_excel(path, sheet_name="column_definitions", engine="openpyxl")
    # normalize whitespace like your validator does
    df = df.map(lambda x: x.strip() if isinstance(x, str) else x)
    return df

Repo root: /Users/indiana-kretzschmar/excel-sbol-test/Excel-to-SBOL
Template: /Users/indiana-kretzschmar/excel-sbol-test/Excel-to-SBOL/resources/templates/Resources.xlsm


In [4]:
p = fresh_copy("baseline")
res = run_v(p, echo=False)
show(res)

# Don't assert ok=True yet — templates often intentionally contain known issues.
# This cell is for "what does baseline do" visibility.

ok: False
errors: 3
warnings: 3
ERROR: COLUMN_DEF_MISSING_IN_SHEET: chassis/PubMed Id — Column is declared in column_definitions but missing from the sheet header. Fix: Add the column to the sheet or remove/fix the row in column_definitions
ERROR: COLUMN_DEF_MISSING_IN_SHEET: rbs/URL Checker — Column is declared in column_definitions but missing from the sheet header. Fix: Add the column to the sheet or remove/fix the row in column_definitions
ERROR: COLUMN_DEF_MISSING_IN_SHEET: signal/PubMed Id — Column is declared in column_definitions but missing from the sheet header. Fix: Add the column to the sheet or remove/fix the row in column_definitions
WARN : UNDECLARED_COLUMN: chassis/PubMed ID — Column exists in sheet but is missing from column_definitions (extra/unexpected column). Fix: Add it to column_definitions or rename/remove the column
WARN : UNDECLARED_COLUMN: cds/Translate to Protein — Column exists in sheet but is missing from column_definitions (extra/unexpected column). Fix: 

In [21]:
import sys
from pathlib import Path
import importlib

# --- find repo root (works even if notebook is in tests/) ---
def find_repo_root(start: Path) -> Path:
    p = start.resolve()
    for _ in range(10):
        if (p / "pyproject.toml").exists():
            return p
        p = p.parent
    raise RuntimeError("Could not find repo root (pyproject.toml not found).")

REPO_ROOT = find_repo_root(Path.cwd())
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# --- always reload validator so edits to validator.py take effect ---
import excel2sbol.validator as v
importlib.reload(v)

# --- set the workbook you are editing ---
RESOURCES = REPO_ROOT / "resources" / "templates" / "Resources.xlsm"
assert RESOURCES.exists(), f"Resources workbook not found: {RESOURCES}"

print("Using workbook:", RESOURCES)
print("Validator from :", Path(v.__file__).resolve())

result = v.run_sheet_validator(str(RESOURCES), validate_only=True, echo=False)

print("\nok:", result["ok"])
print("errors:", len(result["errors"]))
print("warnings:", len(result["warnings"]))

# Print all messages (clear and one-line)
for e in result["errors"]:
    print("ERROR:", e["message"])
for w in result["warnings"]:
    print("WARN :", w["message"])

Using workbook: /Users/indiana-kretzschmar/excel-sbol-test/Excel-to-SBOL/resources/templates/Resources.xlsm
Validator from : /Users/indiana-kretzschmar/excel-sbol-test/Excel-to-SBOL/src/excel2sbol/validator.py

ok: True
errors: 0
warnings: 0


In [22]:
# Run this AFTER you ran the previous cell at least once.

def codeset(res):
    return set((x["code"], x["sheet"], x["column"], x["message"]) for x in (res["errors"] + res["warnings"]))

# store baseline the first time
if "BASELINE" not in globals():
    BASELINE = result
    print("Baseline saved. Now edit Resources.xlsm and rerun this cell.")
else:
    new = result
    added = codeset(new) - codeset(BASELINE)
    removed = codeset(BASELINE) - codeset(new)

    print("NEW items since baseline:", len(added))
    for item in sorted(added):
        code, sheet, col, msg = item
        kind = "ERROR" if any(e["message"] == msg for e in new["errors"]) else "WARN"
        print(f"{kind}:", msg)

    print("\nRESOLVED items since baseline:", len(removed))
    for item in sorted(removed):
        code, sheet, col, msg = item
        print("RESOLVED:", msg)

NEW items since baseline: 0

RESOLVED items since baseline: 6
RESOLVED: COLUMN_DEF_MISSING_IN_SHEET: chassis/PubMed Id — Column is declared in column_definitions but missing from the sheet header. Fix: Add the column to the sheet or remove/fix the row in column_definitions
RESOLVED: COLUMN_DEF_MISSING_IN_SHEET: rbs/URL Checker — Column is declared in column_definitions but missing from the sheet header. Fix: Add the column to the sheet or remove/fix the row in column_definitions
RESOLVED: COLUMN_DEF_MISSING_IN_SHEET: signal/PubMed Id — Column is declared in column_definitions but missing from the sheet header. Fix: Add the column to the sheet or remove/fix the row in column_definitions
RESOLVED: UNDECLARED_COLUMN: cds/Translate to Protein — Column exists in sheet but is missing from column_definitions (extra/unexpected column). Fix: Add it to column_definitions or rename/remove the column
RESOLVED: UNDECLARED_COLUMN: chassis/PubMed ID — Column exists in sheet but is missing from column